In [ ]:
# Default parameter values — overridden by papermill at runtime
run_id = "00000000-0000-0000-0000-000000000000"
input_s3_path = "s3://bucket/raw/predictions/1/00000000-0000-0000-0000-000000000000/input.csv"
bucket = "django-prefect-datalake-dev"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"

In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
endpoint_url = f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint
storage_options = {
    "endpoint_url": endpoint_url,
}
s3 = s3fs.S3FileSystem(
    endpoint_url=endpoint_url,
    client_kwargs={"region_name": aws_s3_region},
)

In [ ]:
# Read the 1-row prediction CSV eagerly
df = pl.read_csv(input_s3_path, storage_options=storage_options)

expected_cols = {"income", "age", "credit_score", "employment_years"}
missing = expected_cols - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

print(f"Schema: {df.schema}")
print(f"Rows: {len(df)}")

In [ ]:
output_s3_path = f"s3://{bucket}/processed/flows/credit-prediction/{run_id}/predict_01_raw.parquet"

with s3.open(output_s3_path, "wb") as f:
    df.write_parquet(f, compression="snappy", use_pyarrow=True)

print(f"Ingested {len(df)} rows → {output_s3_path}")
print(json.dumps({"step": "predict_ingest", "row_count": len(df), "output": output_s3_path}))